# 12 — Hitta dubblettbilder i epic_dataset
Använder MD5-hash på filinnehållet (inte filnamn) för att hitta exakta dubbletter, oavsett vad de heter eller var de kommer ifrån. Samma teknik som hittade de 212 CelebA-dubbletterna tidigare i projektet.

In [ ]:
import os
import hashlib
from collections import defaultdict

DATA_DIR = 'data/epic_dataset'

def collect_files(folder):
    paths = []
    for root, dirs, files in os.walk(folder):
        for f in files:
            if f.lower().endswith(('.jpg', '.jpeg', '.png', '.webp')):
                paths.append(os.path.join(root, f))
    return paths

def md5_of_file(path):
    hasher = hashlib.md5()
    with open(path, 'rb') as f:
        hasher.update(f.read())
    return hasher.hexdigest()

all_files = collect_files(DATA_DIR)
print(f'Skannar {len(all_files)} filer...')

## Beräkna hash för alla filer och gruppera

In [ ]:
hash_to_paths = defaultdict(list)

for i, path in enumerate(all_files):
    h = md5_of_file(path)
    hash_to_paths[h].append(path)

    if (i + 1) % 500 == 0:
        print(f'{i + 1}/{len(all_files)} klara...')

duplicate_groups = {h: paths for h, paths in hash_to_paths.items() if len(paths) > 1}

total_duplicate_files = sum(len(paths) - 1 for paths in duplicate_groups.values())

print(f'\n{len(duplicate_groups)} dubblettgrupper hittade.')
print(f'{total_duplicate_files} överflödiga filer (kan tas bort, en kopia per grupp behålls).')

## Viktigast: dubbletter som spänner över OLIKA klasser
Om samma bild ligger i t.ex. både `epic/` och `medium/` är det en motsägelse i etiketteringen — värre än en vanlig dubblett, eftersom modellen får motsägelsefulla instruktioner om exakt samma bild.

In [ ]:
## Visa cross-class-dubbletterna visuellt — bestäm rätt klass manuellt
**Kör inte borttagningscellen i botten på cross-class-grupper.** De kräver ett medvetet beslut om vilken klass som faktiskt är rätt, annars kan du av misstag permanenta en felaktig etikett bara baserat på mappordning.

import matplotlib.pyplot as plt
from PIL import Image

for h, paths, classes in cross_class_groups:
    fig, axes = plt.subplots(1, len(paths), figsize=(4 * len(paths), 4))
    if len(paths) == 1:
        axes = [axes]
    for ax, p in zip(axes, paths):
        img = Image.open(p)
        ax.imshow(img)
        ax.set_title(get_class(p), fontsize=10)
        ax.axis('off')
    plt.suptitle(os.path.basename(paths[0]))
    plt.tight_layout()
    plt.show()

In [ ]:
same_class_groups = [
    (h, paths) for h, paths in duplicate_groups.items()
    if len(set(get_class(p) for p in paths)) == 1
]

print(f'{len(same_class_groups)} dubblettgrupper inom samma klass.')
print('Exempel (första 10 grupperna):\n')
for h, paths in same_class_groups[:10]:
    print(paths)

## Ta bort överflödiga kopior (behåll en per grupp)
**Kör inte denna förrän du granskat grupperna ovan.** Tar bort alla utom den första filen i varje dubblettgrupp (både cross-class och same-class).

In [ ]:
removed_count = 0

# OBS: bara same_class_groups — cross_class_groups rörs INTE här,
# de kräver manuellt beslut (se bilderna ovan) innan något tas bort.
for h, paths in same_class_groups:
    keep, *rest = paths
    for p in rest:
        os.remove(p)
        removed_count += 1

print(f'{removed_count} dubblettfiler borttagna (bara inom samma klass).')
print(f'{len(cross_class_groups)} cross-class-grupper lämnade orörda — hantera manuellt nedan.')

## Lös cross-class-fallen manuellt
För varje grupp ovan, bestäm vilken klass som är rätt baserat på vad du faktiskt ser på bilden. Skriv in dina beslut i listan nedan (sökvägen till filen som ska **behållas** — resten i samma grupp tas bort).

In [ ]:
# Fyll i sökvägen till den bild du vill BEHÅLLA för varje grupp, en per rad.
# Exempel: om gruppen var medium/thin och du bedömer att den är thin,
# skriv in thin-sökvägen här. Resten i gruppen tas bort.
paths_to_keep = [
    # 'data/epic_dataset/thin/11092.JPG',
    # 'data/epic_dataset/medium/076138.jpg',
]

paths_to_keep_set = set(paths_to_keep)

cross_class_removed = 0
for h, paths, classes in cross_class_groups:
    keepers = [p for p in paths if p in paths_to_keep_set]
    if not keepers:
        print(f'VARNING: inget beslut angivet för grupp {paths} — hoppar över.')
        continue
    for p in paths:
        if p not in paths_to_keep_set:
            os.remove(p)
            cross_class_removed += 1

print(f'{cross_class_removed} felaktiga cross-class-kopior borttagna.')